In [1]:
import json                                                                                                                                               
from pathlib import Path                                                                                                                                

trace_path = Path("results/metrics/grid_tfidf+llm_traces.jsonl")
rows = [json.loads(l) for l in trace_path.read_text().splitlines() if l.strip()]

# Print the first 2 raw responses so you can see what the model is producing
for i, r in enumerate(rows[:2]):
    print(f"--- query {r['job_id']} | strategy: {r['parse_strategy']} ---")
    print("Response text:")
    print(r["response_text"][:1500])
    print()
    print(f"(total length: {len(r['response_text'])} chars)")
    print("=" * 80)  

--- query 689 | strategy: lexical_fallback ---
Response text:


(total length: 0 chars)
--- query 759 | strategy: lexical_fallback ---
Response text:


(total length: 0 chars)


In [ ]:
import requests                                                                                                                                           
from src.config import LM_STUDIO_BASE_URL                                                                                                                 

# Test 1: trivial prompt to confirm the API works at all                                                                                                  
r = requests.post(
    f"{LM_STUDIO_BASE_URL}/chat/completions",                                                                                                             
    json={                                                                                                                                                
        "model": "google/gemma-4-e2b",
        "messages": [{"role": "user", "content": "Say hello in 5 words."}],                                                                               
        "max_tokens": 50,                                                                                                                                 
        "temperature": 0.0,                                                                                                                               
    },                                                                                                                                                    
    timeout=30,                                                                                                                                           
)               
print("=== Trivial prompt test ===")
print(f"Status: {r.status_code}")
payload = r.json()                                                                                                                                        
print(f"Content: {repr(payload['choices'][0]['message']['content'])}")
print(f"Finish reason: {payload['choices'][0].get('finish_reason')}")                                                                                     
print(f"Usage: {payload.get('usage')}")

=== Trivial prompt test ===
Status: 200
Content: ''
Finish reason: length
Usage: {'prompt_tokens': 23, 'completion_tokens': 50, 'total_tokens': 73, 'completion_tokens_details': {'reasoning_tokens': 47}}


In [3]:
import requests                                                                                                                                           
from src.config import LM_STUDIO_BASE_URL                                                                                                                 
                                                                                                                                                        
r = requests.post(                                                                                                                                        
    f"{LM_STUDIO_BASE_URL}/chat/completions",
    json={
        "model": "google/gemma-4-e2b",                                                                                                                    
        "messages": [{"role": "user", "content": "Say hello in 5 words."}],
        "max_tokens": 50,                                                                                                                                 
        "temperature": 0.0,
        "reasoning_effort": "minimal",                                                                                                                    
    },
    timeout=30,                                                                                                                                           
)               
payload = r.json()
print("Content:", repr(payload['choices'][0]['message']['content']))
print("Finish:", payload['choices'][0].get('finish_reason'))                                                                                              
print("Usage:", payload.get('usage'))

Content: ''
Finish: length
Usage: {'prompt_tokens': 23, 'completion_tokens': 50, 'total_tokens': 73, 'completion_tokens_details': {'reasoning_tokens': 47}}


In [4]:
import requests                                                                                                                                           
from src.config import LM_STUDIO_BASE_URL

# Variation A: nested reasoning param (OpenAI o1 style)                                                                                                   
r = requests.post(
    f"{LM_STUDIO_BASE_URL}/chat/completions",                                                                                                             
    json={      
        "model": "google/gemma-4-e2b",                                                                                                                    
        "messages": [
            {"role": "system", "content": "Output the answer directly. Do not show any reasoning or chain of thought."},                                  
            {"role": "user", "content": "Say hello in 5 words."}                                                                                          
        ],                                                                                                                                                
        "max_tokens": 50,                                                                                                                                 
        "temperature": 0.0,
        "reasoning": {"effort": "minimal"},
        "thinking": False,                                                                                                                                
    },
    timeout=30,                                                                                                                                           
)               
payload = r.json()
print("Content:", repr(payload['choices'][0]['message']['content']))
print("Reasoning tokens:", payload['usage']['completion_tokens_details']['reasoning_tokens'])   

Content: 'Hello there, how are you?'
Reasoning tokens: 0


In [6]:
import requests                                                                                                                                           
from src.config import LM_STUDIO_BASE_URL                                                                                                               
r = requests.post(                                                                                                                                        
    f'{LM_STUDIO_BASE_URL}/chat/completions',
    json={                                                                                                                                                
        'model': 'auto',  # LM Studio uses currently loaded model                                                                                       
        'messages': [                                                                                                                                     
            {'role': 'system', 'content': 'You rerank resume candidates. Output JSON only.'},
            {'role': 'user', 'content': '{"task":"Rank these.","candidates":[{"resume_id":"A"},{"resume_id":"B"},{"resume_id":"C"}]}'},                   
        ],                                                                                                                                                
        'max_tokens': 200,                                                                                                                                
        'temperature': 0.0,                                                                                                                               
    },                                                                                                                                                  
    timeout=60,
)
payload = r.json()
print('Content:', payload['choices'][0]['message']['content'])                                                                                            
print('Usage:', payload['usage'])

Content: {"ranking": ["B", "C", "A"]}
Usage: {'prompt_tokens': 47, 'completion_tokens': 13, 'total_tokens': 60, 'completion_tokens_details': {'reasoning_tokens': 0}}


In [9]:
import json, pandas as pd
from pathlib import Path
trace_path = Path("results/metrics/grid_tfidf+llm_traces.jsonl")
if trace_path.exists():
    rows = [json.loads(l) for l in trace_path.read_text().splitlines() if l.strip()]
    print(f"{len(rows)} traces so far")
    print(pd.Series([r['parse_strategy'] for r in rows]).value_counts())

1 traces so far
direct_json    1
Name: count, dtype: int64


In [12]:
import json                                                                                                                                               
from pathlib import Path
rows = [json.loads(l) for l in Path("results/metrics/grid_tfidf+llm_traces.jsonl").read_text().splitlines() if l.strip()]
                                                                                                                                                        
# Average latency vs output size                                                                                                                          
print(f"Traces so far: {len(rows)}")                                                                                                                      
print(f"Avg latency:   {sum(r['latency_seconds'] for r in rows)/len(rows):.1f} s/query")                                                                  
print(f"Avg response:  {sum(len(r['response_text']) for r in rows)/len(rows):.0f} chars")                                                                 
print()                                                                                                                                                   
print("Sample response:")                                                                                                                                 
print(rows[-1]['response_text'][:500])  

Traces so far: 86
Avg latency:   35.6 s/query
Avg response:  881 chars

Sample response:
{
"ranked_candidates": [
{"resume_id": 13855004, "score": 27},
{"resume_id": 19302310, "score": 26},
{"resume_id": 15265464, "score": 25},
{"resume_id": 66906212, "score": 24},
{"resume_id": 28325193, "score": 23},
{"resume_id": 21773106, "score": 22},
{"resume_id": 21550454, "score": 21},
{"resume_id": 13296856, "score": 20},
{"resume_id": 14722634, "score": 19},
{"resume_id": 27710853, "score": 18},
{"resume_id": 68781345, "score": 17},
{"resume_id": 25451319, "score": 16},
{"resume_id": 13593
